# 🩺 AI Health Report Analyzer & Personal Health Assistant
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arun110908/ai-health-report-analyzer/blob/main/notebooks/AI_Health_Report_Analyzer_Demo.ipynb)

A multi-agent pipeline that turns a raw blood report (PDF / image / DOCX / text) into a simple, structured health summary.

| Agent | Job | Uses LLM? |
|---|---|---|
| **1. Report Processing** | OCR / text extraction, parameter parsing, name + unit normalisation, low/normal/high classification | No (deterministic) |
| **2. Medical Analysis** | Finds deficiencies and risks, explains abnormal values, health score | Claude (optional) + rule-based fallback |
| **3. Recommendation** | Diet, exercise, hydration, sleep, lifestyle, follow-up | Claude (optional) + rule-based fallback |
| **4. Report Generation** | Assembles the final structured report | No (deterministic) |

Orchestrated with **LangGraph**. Works **without any API key** (rule-based fallback); add `ANTHROPIC_API_KEY` in Colab Secrets to switch Agents 2 and 3 to Claude.

> ⚠️ Informational tool, **not a medical diagnosis**. All demo data here is synthetic.

**How to run:** Runtime → **Run all**. The first cell takes ~1-2 minutes (installs Tesseract + dependencies).

## 1. Setup

In [ ]:
#@title Clone the repo and install dependencies { display-mode: "form" }
import os, shutil

REPO_URL = "https://github.com/Arun110908/ai-health-report-analyzer.git"
if not os.path.isdir("ai-health-report-analyzer") and not os.getcwd().endswith("ai-health-report-analyzer"):
    !git clone -q $REPO_URL
if not os.getcwd().endswith("ai-health-report-analyzer"):
    %cd ai-health-report-analyzer

if shutil.which("tesseract") is None:
    !apt-get -qq install -y tesseract-ocr poppler-utils > /dev/null

!pip install -q -r backend/requirements.txt
print("\n✅ Setup done (pip may print dependency-conflict warnings above; they are harmless here)")

## 2. Load the pipeline
Optional: add a Colab Secret named `ANTHROPIC_API_KEY` (🔑 icon in the left sidebar) to use Claude in Agents 2 and 3. **Never paste the key into a cell.**

In [ ]:
import os, sys, re, glob
sys.path.insert(0, "backend")

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("🔑 Claude key loaded from Colab Secrets")
except Exception:
    print("ℹ️ No ANTHROPIC_API_KEY secret found -> rule-based mode (everything still works)")

from app.models import PipelineState
from app.pipeline import run_pipeline, using_langgraph
from app.llm_client import claude_client, MODEL_NAME

print("Pipeline engine:", "LangGraph" if using_langgraph() else "sequential fallback")
print("Claude enabled :", claude_client.available, f"({MODEL_NAME})")

## 3. Helper functions (display, chart)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

STATUS_COLOR = {"normal": "#2e7d32", "low": "#ef6c00", "high": "#ef6c00",
                "critical_low": "#c62828", "critical_high": "#c62828", "unknown": "#757575"}

def _color(col):
    return [f"color: {STATUS_COLOR.get(v, '#000')}; font-weight: bold" for v in col]

def parse_range(s):
    if not s:
        return None, None
    m = re.match(r"^\s*([\d.]+)\s*-\s*([\d.]+)\s*$", s)
    if m:
        return float(m.group(1)), float(m.group(2))
    m = re.match(r"^\s*<\s*([\d.]+)\s*$", s)
    if m:
        return None, float(m.group(1))
    m = re.match(r"^\s*>\s*([\d.]+)\s*$", s)
    if m:
        return float(m.group(1)), None
    return None, None

def plot_ranges(params):
    rows = []
    for p in params:
        lo, hi = parse_range(p.reference_range)
        if lo is None or hi is None or p.value is None or hi <= lo:
            continue
        pos = (p.value - lo) / (hi - lo)
        rows.append((p.name, max(min(pos, 1.7), -0.7), p.status))
    if not rows:
        return
    fig, ax = plt.subplots(figsize=(8, 0.38 * len(rows) + 1.2))
    ax.axvspan(0, 1, color="#c8e6c9", alpha=0.7)
    for i, (name, pos, status) in enumerate(rows):
        ax.scatter(pos, i, s=70, color=STATUS_COLOR.get(status, "#757575"), zorder=3)
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([r[0] for r in rows])
    ax.invert_yaxis()
    ax.set_xlim(-0.8, 1.8)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["low limit", "high limit"])
    ax.set_title("Each value vs its normal range (green band = normal)")
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.show()

def analyze_text(text, file_type="text", ocr_confidence=1.0):
    return run_pipeline(PipelineState(raw_text=text, file_type=file_type, ocr_confidence=ocr_confidence))

def show(state):
    r = state.final_report
    if r is None:
        print("Pipeline failed:", state.errors)
        return
    display(Markdown(f"## Health score: **{r.overall_health_score}/100**\n\n{r.summary}"))
    if r.critical_alerts:
        display(Markdown("### 🚨 Critical alerts\n" + "\n".join(f"- {a}" for a in r.critical_alerts)))
    if r.doctor_consultation_suggested:
        display(Markdown(f"### 🩺 Doctor consultation suggested\n{r.doctor_consultation_reason or ''}"))
    df = pd.DataFrame([{"Parameter": p.name, "Value": p.value, "Unit": p.unit,
                        "Reference": p.reference_range, "Status": p.status} for p in r.parameters])
    display(df.style.apply(_color, subset=["Status"]).hide(axis="index"))
    plot_ranges(r.parameters)
    if r.deficiencies:
        display(Markdown("### Deficiencies\n" + "\n".join(f"- **{d.title}** ({d.severity}): {d.explanation}" for d in r.deficiencies)))
    if r.health_risks:
        display(Markdown("### Health risks\n" + "\n".join(f"- **{d.title}** ({d.severity}): {d.explanation}" for d in r.health_risks)))
    rec = r.recommendations
    for label, items in [("🥗 Eat", rec.diet_eat), ("🚫 Avoid", rec.diet_avoid), ("🏃 Exercise", rec.exercise),
                         ("💧 Hydration", rec.hydration), ("😴 Sleep", rec.sleep),
                         ("🧘 Stress", rec.stress_management), ("🌿 Lifestyle", rec.lifestyle)]:
        if items:
            display(Markdown(f"**{label}**\n" + "\n".join(f"- {i}" for i in items)))

def show_agents(state):
    e, a, rec, f = state.extracted, state.analysis, state.recommendations, state.final_report
    print("Agent 1 · Report Processing :", len(e.parameters), "parameters | format:", e.report_format_detected,
          "| OCR confidence:", e.ocr_confidence, "| warnings:", len(e.warnings))
    for w in e.warnings[:6]:
        print("      ⚠", w)
    print("Agent 2 · Medical Analysis  :", len(a.abnormal_parameters), "abnormal ->", a.abnormal_parameters,
          "| deficiencies:", len(a.deficiencies), "| risks:", len(a.health_risks))
    n = sum(len(x) for x in (rec.diet_eat, rec.diet_avoid, rec.exercise, rec.hydration,
                             rec.sleep, rec.stress_management, rec.lifestyle))
    print("Agent 3 · Recommendations   :", n, "recommendation items")
    print("Agent 4 · Report Generation : health score", f.overall_health_score,
          "| doctor consultation:", f.doctor_consultation_suggested)
print("✅ helpers ready")

## 4. Demo: analyze a sample report
`backend/sample_data/` has 25 **synthetic** reports in 4 different layouts. Change the file name below to try others (`sample_report_01.txt` ... `sample_report_25.txt`).

In [ ]:
#@title Pick a sample report { display-mode: "form" }
SAMPLE = "sample_report_03.txt" #@param {type:"string"}

text = open(f"backend/sample_data/{SAMPLE}").read()
print("\n".join(text.splitlines()[:22]))
print("   ...")

In [ ]:
state = analyze_text(text)
show(state)

### What each agent produced
This is the shared `PipelineState` after the LangGraph run: one output per agent.

In [ ]:
show_agents(state)

## 5. Demo: messy real-world lab formats
`backend/tests_realistic/` has hand-written reports that imitate real lab layouts: a **technology column**, **SI units** (mmol/L, µmol/L, g/L), `cells/cumm`, `lakhs/cumm`, OCR-style typos, and ranges split across lines. Agent 1 converts units and classifies using the range **printed on the report**.

In [ ]:
messy = open("backend/tests_realistic/r3_si_units.txt").read()
print(messy)

In [ ]:
state2 = analyze_text(messy)
print("Values below are converted to standard units (e.g. glucose mmol/L -> mg/dL)\n")
rows = [{"Parameter": p.name, "Raw": p.raw_value, "Value (std)": p.value, "Unit": p.unit,
         "Reference": p.reference_range, "Status": p.status} for p in state2.extracted.parameters]
display(pd.DataFrame(rows).style.apply(_color, subset=["Status"]).hide(axis="index"))
for w in state2.extracted.warnings:
    print("⚠", w)

## 6. Evaluation against ground truth
`scripts/evaluate.py` runs the pipeline on labelled reports and measures extraction recall, value accuracy, and low/normal/high accuracy.

> ⚠️ Both sets are **synthetic / hand-written**, so ~100% here does **not** mean 100% on real reports. Real-report validation is the next step.

In [ ]:
print("=== 25 synthetic reports, 4 layouts ===")
!python backend/scripts/evaluate.py 2>/dev/null | head -9
print("\n=== 5 hand-written realistic formats ===")
!python backend/scripts/evaluate.py --dir backend/tests_realistic 2>/dev/null | head -9

## 7. (Optional) Try your own report
Tick the box and run the cell to upload a `.pdf`, `.png/.jpg`, `.docx` or `.txt`. **Use de-identified reports only** (remove name, ID, phone). Scanned images go through Tesseract OCR; if OCR confidence is below 85% the report shows a "please verify" warning.

In [ ]:
#@title Upload a report { display-mode: "form" }
RUN_UPLOAD = False #@param {type:"boolean"}

if RUN_UPLOAD:
    from google.colab import files
    from app.ocr_extraction import extract_text_from_pdf, extract_text_from_image, extract_text_from_docx
    uploaded = files.upload()
    for fname, data in uploaded.items():
        ext = fname.lower().rsplit(".", 1)[-1]
        if ext == "pdf":
            txt, conf = extract_text_from_pdf(data)
        elif ext in ("png", "jpg", "jpeg"):
            txt, conf = extract_text_from_image(data)
        elif ext == "docx":
            txt, conf = extract_text_from_docx(data)
        else:
            txt, conf = data.decode("utf-8", errors="ignore"), 1.0
        print(f"{fname}: extracted {len(txt)} characters, OCR confidence {conf:.0%}")
        st = analyze_text(txt, file_type=ext, ocr_confidence=conf)
        for w in st.extracted.warnings:
            print("⚠", w)
        show(st)
else:
    print("Tick RUN_UPLOAD above, then run this cell again.")

## 8. Summary and limitations

**Design choices**
- Extraction and flagging (Agent 1) are **deterministic and testable**. The LLM is only used for explanation and advice (Agents 2 and 3), with Pydantic-validated JSON and a rule-based fallback.
- A curated reference-range table (WHO / NIH / Mayo summaries) grounds the LLM prompts and is the fallback when a report has no printed range. It is a curated lookup, not a vector-search RAG.
- The range **printed on the report** is preferred over the table, because normal ranges depend on the lab, sex and age.

**Limitations (be honest)**
- Accuracy numbers are on synthetic / hand-written data, not real patient reports.
- Tesseract accuracy drops on blurry photos; a vision-LLM extractor would help.
- Sex/age-specific ranges are not yet taken from patient info when the report prints no range.
- Informational only, not a diagnosis.

**Next steps:** validate on de-identified real reports, add a Claude-vision extraction path for poor scans, more parameters, authentication for a public deployment.